In [ ]:
"""Manufactured SM2.ipynb

"""

In [ ]:
from __future__ import annotations

In [ ]:
import argparse
import copy
import csv
import json
import math
import os
import random
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Literal, Tuple

In [ ]:
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
plt.rcParams.update({'font.size': 20,
                    'axes.titlesize': 22,
                    'axes.labelsize': 20,
                    'xtick.labelsize': 18,
                    'ytick.labelsize': 18,
                    'legend.fontsize': 18})

In [ ]:
import numpy as np
import torch
from torch import nn

In [ ]:
Tensor = torch.Tensor
Method = Literal["PINN", "CRVPINN"]

In [ ]:
@dataclass
class Config:
    epochs: int = 20000
    n_train: int = 25
    n_eval: int = 25
    width: int = 100
    hidden_layers: int = 2
    learning_rate: float = 2.0e-3
    lr_decay: float = 0.9995
    log_every: int = 50
    plot_points: int = 121
    seed: int = 1234
    dtype: str = "float32"
    device: str = "auto"
    output_dir: str = "advection_diffusion_results"
    cpu_threads: int = 4

In [ ]:
def parse_args() -> Config:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--epochs", type=int, default=Config.epochs)
    parser.add_argument("--n-train", type=int, default=Config.n_train,
                        help="Liczba wewnętrznych punktów na każdej osi.")
    parser.add_argument("--n-eval", type=int, default=Config.n_eval,
                        help="Liczba wewnętrznych punktów na oś do true error.")
    parser.add_argument("--width", type=int, default=Config.width)
    parser.add_argument("--hidden-layers", type=int, default=Config.hidden_layers)
    parser.add_argument("--learning-rate", type=float, default=Config.learning_rate)
    parser.add_argument("--lr-decay", type=float, default=Config.lr_decay)
    parser.add_argument("--log-every", type=int, default=Config.log_every)
    parser.add_argument("--plot-points", type=int, default=Config.plot_points)
    parser.add_argument("--seed", type=int, default=Config.seed)
    parser.add_argument("--dtype", choices=("float32", "float64"), default=Config.dtype)
    parser.add_argument("--device", choices=("auto", "cpu", "cuda"), default=Config.device)
    parser.add_argument("--output-dir", type=str, default=Config.output_dir)
    parser.add_argument("--cpu-threads", type=int, default=Config.cpu_threads,
                        help="Liczba wątków PyTorch na CPU; zbyt duża liczba silnie spowalnia małe sieci.")
    args, unknown = parser.parse_known_args(sys.argv[1:])
    return Config(**vars(args))

In [ ]:
def select_device(name: str) -> torch.device:
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if name == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("Wybrano CUDA, ale torch.cuda.is_available() == False.")
    return torch.device(name)

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
def u_exact(x: Tensor, y: Tensor, t: Tensor) -> Tensor:
    return torch.sin(math.pi * x) * torch.sin(math.pi * y) * torch.exp(-t)

In [ ]:
def initial_condition(x: Tensor, y: Tensor) -> Tensor:
    return torch.sin(math.pi * x) * torch.sin(math.pi * y)

In [ ]:
def forcing(x: Tensor, y: Tensor, t: Tensor) -> Tensor:
    sx = torch.sin(math.pi * x)
    sy = torch.sin(math.pi * y)
    cx = torch.cos(math.pi * x)
    cy = torch.cos(math.pi * y)
    return torch.exp(-t) * (
        (2.0 * math.pi**2 - 1.0) * sx * sy
        + math.pi * cx * sy
        + math.pi * sx * cy
    )

In [ ]:
def exact_derivatives(x: Tensor, y: Tensor, t: Tensor) -> Tuple[Tensor, Tensor, Tensor]:
    exp_t = torch.exp(-t)
    ux = math.pi * torch.cos(math.pi * x) * torch.sin(math.pi * y) * exp_t
    uy = math.pi * torch.sin(math.pi * x) * torch.cos(math.pi * y) * exp_t
    ut = -u_exact(x, y, t)
    return ux, uy, ut

In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden_layers: int, width: int) -> None:
        super().__init__()
        layers: List[nn.Module] = [nn.Linear(3, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_normal_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, x: Tensor, y: Tensor, t: Tensor) -> Tensor:
        raw = self.net(torch.cat((x, y, t), dim=1))
        # Dokładne narzucenie u(x,y,0)=u0 oraz u=0 na brzegu przestrzennym.
        boundary_factor = x * (1.0 - x) * y * (1.0 - y)
        return initial_condition(x, y) + t * boundary_factor * raw

In [ ]:
def interior_grid(n: int, device: torch.device, dtype: torch.dtype,
                  requires_grad: bool = True) -> Tuple[Tensor, Tensor, Tensor, float]:
    if n < 2:
        raise ValueError("n musi być co najmniej 2.")
    h = 1.0 / (n + 1)
    axis = torch.arange(1, n + 1, device=device, dtype=dtype) * h
    x, y, t = torch.meshgrid(axis, axis, axis, indexing="ij")
    x = x.reshape(-1, 1).clone().detach().requires_grad_(requires_grad)
    y = y.reshape(-1, 1).clone().detach().requires_grad_(requires_grad)
    t = t.reshape(-1, 1).clone().detach().requires_grad_(requires_grad)
    return x, y, t, h

In [ ]:
def pde_residual(model: nn.Module, x: Tensor, y: Tensor, t: Tensor) -> Tensor:
    u = model(x, y, t)
    ones = torch.ones_like(u)
    ux, uy, ut = torch.autograd.grad(
        u, (x, y, t), grad_outputs=ones, create_graph=True, retain_graph=True
    )
    uxx = torch.autograd.grad(
        ux, x, grad_outputs=torch.ones_like(ux), create_graph=True, retain_graph=True
    )[0]
    uyy = torch.autograd.grad(
        uy, y, grad_outputs=torch.ones_like(uy), create_graph=True, retain_graph=True
    )[0]
    return ut + ux + uy - uxx - uyy - forcing(x, y, t)

In [ ]:
class SinePoissonInverse3D(nn.Module):
    """Działanie odwrotności 3D macierzy Dirichletowskiego Laplasjanu.

    A = kron(I,I,L) + kron(I,L,I) + kron(L,I,I),
    gdzie L ma 2 na przekątnej i -1 na sąsiednich przekątnych.
    Macierz Grama z seminormy H1 w 3D ma postać G = h*A.
    """

    def __init__(self, n: int, h: float, device: torch.device, dtype: torch.dtype) -> None:
        super().__init__()
        idx = torch.arange(1, n + 1, device=device, dtype=dtype)
        q = torch.sqrt(torch.tensor(2.0 / (n + 1), device=device, dtype=dtype)) * torch.sin(
            math.pi * idx[:, None] * idx[None, :] / (n + 1)
        )
        lam = 2.0 - 2.0 * torch.cos(math.pi * idx / (n + 1))
        eig = lam[:, None, None] + lam[None, :, None] + lam[None, None, :]
        self.n = n
        self.h = float(h)
        self.register_buffer("q", q)
        self.register_buffer("eig", eig)

    @staticmethod
    def _mode_product(x: Tensor, matrix: Tensor, axis: int) -> Tensor:
        moved = torch.movedim(x, axis, 0)
        shape = moved.shape
        transformed = matrix @ moved.reshape(shape[0], -1)
        transformed = transformed.reshape((matrix.shape[0],) + shape[1:])
        return torch.movedim(transformed, 0, axis)

    def solve_A(self, rhs: Tensor) -> Tensor:
        cube = rhs.reshape(self.n, self.n, self.n)
        spectral = cube
        for axis in range(3):
            spectral = self._mode_product(spectral, self.q.T, axis)
        spectral = spectral / self.eig
        solution = spectral
        for axis in range(3):
            solution = self._mode_product(solution, self.q, axis)
        return solution.reshape(-1, 1)

    def robust_loss_from_strong_residual(self, residual: Tensor) -> Tensor:
        # RES = h^3*r, G = h*A, więc RES^T G^{-1} RES = h^5*r^T A^{-1}r.
        a_inv_r = self.solve_A(residual)
        loss = (self.h**5) * torch.sum(residual * a_inv_r)
        return torch.clamp(loss, min=0.0)

In [ ]:
def compute_losses(model: nn.Module, x: Tensor, y: Tensor, t: Tensor,
                   gram_inverse: SinePoissonInverse3D) -> Tuple[Tensor, Tensor, Tensor]:
    residual = pde_residual(model, x, y, t)
    pinn_loss = torch.mean(residual.square())
    crvpinn_loss = gram_inverse.robust_loss_from_strong_residual(residual)
    return pinn_loss, crvpinn_loss, residual

In [ ]:
def true_errors(model: nn.Module, n: int, device: torch.device,
                dtype: torch.dtype) -> Tuple[float, float]:
    x, y, t, h = interior_grid(n, device, dtype, requires_grad=True)
    u = model(x, y, t)
    ones = torch.ones_like(u)
    ux, uy, ut = torch.autograd.grad(
        u, (x, y, t), grad_outputs=ones, create_graph=False, retain_graph=True
    )
    ue = u_exact(x, y, t)
    uex, uey, uet = exact_derivatives(x, y, t)

    h1_sq = h**3 * torch.sum((ux - uex).square() + (uy - uey).square() + (ut - uet).square())
    l2_sq = h**3 * torch.sum((u - ue).square())
    exact_l2_sq = h**3 * torch.sum(ue.square())
    h1_error = torch.sqrt(torch.clamp(h1_sq, min=0.0)).item()
    relative_l2 = torch.sqrt(torch.clamp(l2_sq / exact_l2_sq, min=0.0)).item()
    return h1_error, relative_l2

In [ ]:
def evaluate_metrics(model: nn.Module, x: Tensor, y: Tensor, t: Tensor,
                     gram_inverse: SinePoissonInverse3D, n_eval: int,
                     device: torch.device, dtype: torch.dtype) -> Dict[str, float]:
    model.eval()
    # Residuum wymaga autograd, dlatego nie używamy torch.no_grad().
    pinn_loss, crv_loss, _ = compute_losses(model, x, y, t, gram_inverse)
    h1_error, relative_l2 = true_errors(model, n_eval, device, dtype)
    return {
        "pinn_loss": float(pinn_loss.detach().cpu()),
        "crvpinn_loss": float(crv_loss.detach().cpu()),
        "pinn_residual_rmse": float(torch.sqrt(torch.clamp(pinn_loss, min=0.0)).detach().cpu()),
        "crvpinn_estimator": float(torch.sqrt(torch.clamp(crv_loss, min=0.0)).detach().cpu()),
        "true_h1_error": h1_error,
        "relative_l2_error": relative_l2,
    }

In [ ]:
def train(model: nn.Module, method: Method, x: Tensor, y: Tensor, t: Tensor,
          gram_inverse: SinePoissonInverse3D, cfg: Config, device: torch.device,
          dtype: torch.dtype, n_train_value: int) -> List[Dict[str, float]]:
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=cfg.lr_decay)
    history: List[Dict[str, float]] = []
    start = time.perf_counter()

    def log(epoch: int) -> None:
        metrics = evaluate_metrics(model, x, y, t, gram_inverse, cfg.n_eval, device, dtype)
        metrics.update({
            "model": method,
            "n_train": n_train_value, # Add n_train_value here
            "epoch": epoch,
            "seconds": time.perf_counter() - start,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "train_objective": metrics["pinn_loss"] if method == "PINN" else metrics["crvpinn_loss"],
        })
        history.append(metrics)
        print(
            f"[{method:8s}, n_train={n_train_value:3d}] epoch={epoch:6d}  "
            f"sqrt(train loss)={math.sqrt(max(metrics['train_objective'], 0.0)):.4e}  "
            f"true H1={metrics['true_h1_error']:.4e}  "
            f"rel L2={metrics['relative_l2_error']:.4e}"
        )

    log(0)
    model.train()
    for epoch in range(1, cfg.epochs + 1):
        optimizer.zero_grad(set_to_none=True)
        pinn_loss, crv_loss, _ = compute_losses(model, x, y, t, gram_inverse)
        objective = pinn_loss if method == "PINN" else crv_loss
        if not torch.isfinite(objective):
            raise FloatingPointError(f"Niefinitywna funkcja celu dla {method}, epoka {epoch}.")
        objective.backward()
        optimizer.step()
        scheduler.step()

        if epoch % cfg.log_every == 0 or epoch == cfg.epochs:
            log(epoch)
            model.train()

    return history

In [ ]:
def save_history(all_histories: List[Dict[str, float]], output_dir: Path) -> None:
    fieldnames = [
        "model", "n_train", "epoch", "seconds", "learning_rate", "train_objective",
        "pinn_loss", "crvpinn_loss", "pinn_residual_rmse", "crvpinn_estimator",
        "true_h1_error", "relative_l2_error",
    ]
    with (output_dir / "history.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_histories)

In [ ]:
def history_arrays(history: List[Dict[str, float]]) -> Dict[str, np.ndarray]:
    keys = history[0].keys()
    return {key: np.asarray([row[key] for row in history]) for key in keys if key not in ("model", "n_train")}

In [ ]:
def plot_convergence_single(histories: Dict[Method, List[Dict[str, float]]], output_dir: Path, n_train_value: int) -> None:
    hp = history_arrays(histories["PINN"])
    hc = history_arrays(histories["CRVPINN"])

    fig, ax = plt.subplots(figsize=(8, 6), dpi=140)
    ax.semilogy(hp["epoch"], hp["pinn_residual_rmse"], label=r"PINN: $\sqrt{\mathrm{MSE}(R)}$")
    ax.semilogy(hp["epoch"], hp["true_h1_error"], label=r"PINN: true $|u-u_\theta|_{H^1_h}$")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Value")
    ax.set_title(f"PINN: Loss vs True Error (n_train={n_train_value})")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / f"convergence_PINN_{n_train_value}.png")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 6), dpi=140)
    ax.semilogy(hc["epoch"], hc["crvpinn_estimator"], label=r"CRVPINN: $\sqrt{\mathrm{LOSS}_{robust}}$")
    ax.semilogy(hc["epoch"], hc["true_h1_error"], label=r"CRVPINN: true $|u-u_\theta|_{H^1_h}$")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Value")
    ax.set_title(f"CRVPINN: Robust Estimator vs True Error (n_train={n_train_value})")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / f"convergence_CRVPINN_{n_train_value}.png")
    plt.close(fig)

In [ ]:
def plot_all_convergences(all_histories: List[Dict[str, float]], output_dir: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 7), dpi=140)
    unique_n_train = sorted(list(set([h['n_train'] for h in all_histories])))

    for n_train_value in unique_n_train:
        pinn_hist = [h for h in all_histories if h['model'] == 'PINN' and h['n_train'] == n_train_value]
        crvpinn_hist = [h for h in all_histories if h['model'] == 'CRVPINN' and h['n_train'] == n_train_value]

        if pinn_hist:
            hp = history_arrays(pinn_hist)
            ax.semilogy(hp["epoch"], hp["true_h1_error"], label=f"PINN (n_train={n_train_value})")
        if crvpinn_hist:
            hc = history_arrays(crvpinn_hist)
            ax.semilogy(hc["epoch"], hc["true_h1_error"], label=f"CRVPINN (n_train={n_train_value})")

    ax.set_xlabel("Epoch")
    ax.set_ylabel(r"True $H^1_h$ error")
    ax.set_title("Comparison of True $H^1_h$ Error Convergence")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "true_h1_error_comparison_all_ntrain.png")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 7), dpi=140)
    for n_train_value in unique_n_train:
        pinn_hist = [h for h in all_histories if h['model'] == 'PINN' and h['n_train'] == n_train_value]
        crvpinn_hist = [h for h in all_histories if h['model'] == 'CRVPINN' and h['n_train'] == n_train_value]

        if pinn_hist:
            hp = history_arrays(pinn_hist)
            ax.semilogy(hp["epoch"], hp["relative_l2_error"], label=f"PINN (n_train={n_train_value})")
        if crvpinn_hist:
            hc = history_arrays(crvpinn_hist)
            ax.semilogy(hc["epoch"], hc["relative_l2_error"], label=f"CRVPINN (n_train={n_train_value})")

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Relative L2 error")
    ax.set_title("Comparison of Relative L2 Error Convergence")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "relative_l2_error_comparison_all_ntrain.png")
    plt.close(fig)

In [ ]:
def plot_loss_to_true_error_ratio(all_histories: List[Dict[str, float]], output_dir: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 7), dpi=140)
    unique_n_train = sorted(list(set([h['n_train'] for h in all_histories])))
    eps = np.finfo(float).tiny

    for n_train_value in unique_n_train:
        pinn_hist = [h for h in all_histories if h['model'] == 'PINN' and h['n_train'] == n_train_value]
        crvpinn_hist = [h for h in all_histories if h['model'] == 'CRVPINN' and h['n_train'] == n_train_value]

        if pinn_hist:
            hp = history_arrays(pinn_hist)
            ax.semilogy(hp["epoch"], hp["pinn_residual_rmse"] / np.maximum(hp["true_h1_error"], eps),
                        label=f"PINN (n_train={n_train_value})")
        if crvpinn_hist:
            hc = history_arrays(crvpinn_hist)
            ax.semilogy(hc["epoch"], hc["crvpinn_estimator"] / np.maximum(hc["true_h1_error"], eps),
                        label=f"CRVPINN (n_train={n_train_value})")

    ax.axhline(1.0, linewidth=1.0, linestyle="--", color="gray", alpha=0.7)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Ratio")
    ax.set_title("Loss to True Error Ratio")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_dir / "loss_to_true_error_ratio_all_ntrain.png")
    plt.close(fig)

In [ ]:
def plot_convergence(histories: Dict[Method, List[Dict[str, float]]], output_dir: Path, n_train_value: int) -> None:
    plot_convergence_single(histories, output_dir, n_train_value)

In [ ]:
def evaluate_slice(model: nn.Module, t_value: float, points: int,
                   device: torch.device, dtype: torch.dtype) -> Tuple[np.ndarray, ...]:
    axis = torch.linspace(0.0, 1.0, points, device=device, dtype=dtype)
    x, y = torch.meshgrid(axis, axis, indexing="ij")
    xf = x.reshape(-1, 1)
    yf = y.reshape(-1, 1)
    tf = torch.full_like(xf, t_value)
    with torch.no_grad():
        pred = model(xf, yf, tf).reshape(points, points)
        exact = u_exact(xf, yf, tf).reshape(points, points)
    error = torch.abs(pred - exact)
    return (
        x.detach().cpu().numpy(), y.detach().cpu().numpy(),
        exact.detach().cpu().numpy(), pred.detach().cpu().numpy(), error.detach().cpu().numpy(),
    )

In [ ]:
def plot_solution_slices(model: nn.Module, method: Method, output_dir: Path,
                         points: int, device: torch.device, dtype: torch.dtype, n_train_value: int) -> None:
    times = (0.25, 0.50, 1.00)
    fig, axes = plt.subplots(len(times), 3, figsize=(13, 11), dpi=140, constrained_layout=True)
    for row, t_value in enumerate(times):
        x, y, exact, pred, error = evaluate_slice(model, t_value, points, device, dtype)
        fields = (exact, pred, error)
        titles = ("exact", method, "|error|")
        for col, (field, title) in enumerate(zip(fields, titles)):
            mesh = axes[row, col].pcolormesh(x, y, field, shading="auto")
            axes[row, col].set_aspect("equal")
            axes[row, col].set_xlabel("x")
            axes[row, col].set_ylabel("y")
            axes[row, col].set_title(f"{title}, t={t_value:.2f}")
            fig.colorbar(mesh, ax=axes[row, col], shrink=0.82)
    fig.suptitle(f"Solution and Error - {method} (n_train={n_train_value})")
    fig.savefig(output_dir / f"solution_slices_{method}_{n_train_value}.png")
    plt.close(fig)

In [ ]:
def plot_centerline(models: Dict[Method, nn.Module], output_dir: Path,
                    device: torch.device, dtype: torch.dtype, n_train_value: int) -> None:
    x = torch.linspace(0.0, 1.0, 401, device=device, dtype=dtype).reshape(-1, 1)
    y = torch.full_like(x, 0.5)
    times = (0.25, 0.50, 1.00)
    fig, axes = plt.subplots(1, len(times), figsize=(15, 4.2), dpi=140, constrained_layout=True)
    for ax, t_value in zip(axes, times):
        t = torch.full_like(x, t_value)
        with torch.no_grad():
            exact = u_exact(x, y, t).cpu().numpy()
            pred_pinn = models["PINN"](x, y, t).cpu().numpy()
            pred_crv = models["CRVPINN"](x, y, t).cpu().numpy()
        x_np = x.cpu().numpy()
        ax.plot(x_np, exact, label="exact", linewidth=2.0)
        ax.plot(x_np, pred_pinn, label="PINN")
        ax.plot(x_np, pred_crv, label="CRVPINN")
        ax.set_xlabel("x")
        ax.set_ylabel("u(x,0.5,t)")
        ax.set_title(f"t={t_value:.2f}")
        ax.grid(True, alpha=0.3)
    axes[-1].legend()
    fig.suptitle(f"Centerline Comparison (y=0.5) - n_train={n_train_value}")
    fig.savefig(output_dir / f"centerline_comparison_{n_train_value}.png")
    plt.close(fig)

In [ ]:
def main() -> None:
    print("This main function is no longer the primary entry point. Please run the notebook cells sequentially.")

In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
"""### Experiment Setup

We will now run the PINN and CRVPINN models for various `n_train` values to observe their convergence behavior.

The `n_train` values to be tested are: 12, 25, 50, and 100.

For each `n_train` value, the following plots will be generated:
- **Individual Convergence Plots**: Showing `pinn_residual_rmse` and `crvpinn_estimator` against `true_h1_error` for both PINN and CRVPINN.

Additionally, comparative plots will be generated:
- **Overall True H1 Error Convergence**: Comparing the `true_h1_error` for PINN and CRVPINN across all `n_train` values.
- **Overall Relative L2 Error Convergence**: Comparing the `relative_l2_error` for PINN and CRVPINN across all `n_train` values.
- **Loss to True Error Ratio**: Comparing the ratio of loss (RMSE for PINN, estimator for CRVPINN) to true H1 error for all `n_train` values.

A combined `history.csv` file will also be saved, containing all training metrics for all runs.

All generated files will be automatically downloaded.
"""

In [ ]:
n_train_values = [12, 25, 50, 100]
all_experiment_histories: List[Dict[str, float]] = []
all_download_files: List[str] = []

In [ ]:
initial_cfg = parse_args()

In [ ]:
for n_train_val in n_train_values:
    print(f"\n{'='*50}")
    print(f"Starting experiment for n_train = {n_train_val}")
    print(f"{'='*50}")

    # Create a configuration for the current n_train value
    cfg = copy.deepcopy(initial_cfg)
    cfg.n_train = n_train_val
    cfg.output_dir = f"advection_diffusion_results_n{n_train_val}"

    set_seed(cfg.seed)
    device = select_device(cfg.device)
    if device.type == "cpu":
        torch.set_num_threads(max(1, cfg.cpu_threads))
    dtype = torch.float32 if cfg.dtype == "float32" else torch.float64
    torch.set_default_dtype(dtype)

    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    with (output_dir / "config.json").open("w", encoding="utf-8") as handle:
        json.dump(asdict(cfg), handle, indent=2, ensure_ascii=False)

    print(f"Device: {device}; dtype: {dtype}; output: {output_dir.resolve()}")
    x, y, t, h = interior_grid(cfg.n_train, device, dtype, requires_grad=True)
    gram_inverse = SinePoissonInverse3D(cfg.n_train, h, device, dtype).to(device)

    base_model = MLP(cfg.hidden_layers, cfg.width).to(device=device, dtype=dtype)
    initial_state = copy.deepcopy(base_model.state_dict())
    models: Dict[Method, nn.Module] = {
        "PINN": MLP(cfg.hidden_layers, cfg.width).to(device=device, dtype=dtype),
        "CRVPINN": MLP(cfg.hidden_layers, cfg.width).to(device=device, dtype=dtype),
    }
    models["PINN"].load_state_dict(initial_state)
    models["CRVPINN"].load_state_dict(initial_state)

    histories: Dict[Method, List[Dict[str, float]]] = {
        "PINN": train(models["PINN"], "PINN", x, y, t, gram_inverse, cfg, device, dtype, n_train_val),
        "CRVPINN": train(models["CRVPINN"], "CRVPINN", x, y, t, gram_inverse, cfg, device, dtype, n_train_val),
    }
    all_experiment_histories.extend(histories["PINN"])
    all_experiment_histories.extend(histories["CRVPINN"])

    torch.save(models["PINN"].state_dict(), output_dir / "model_PINN.pt")
    torch.save(models["CRVPINN"].state_dict(), output_dir / "model_CRVPINN.pt")

    # Generate individual convergence plots for current n_train_val
    plot_convergence(histories, output_dir, n_train_val)

    # Solution slices and centerline plots for current n_train_val
    plot_solution_slices(models["PINN"], "PINN", output_dir, cfg.plot_points, device, dtype, n_train_val)
    plot_solution_slices(models["CRVPINN"], "CRVPINN", output_dir, cfg.plot_points, device, dtype, n_train_val)
    plot_centerline(models, output_dir, device, dtype, n_train_val)

    summary = {
        method: histories[method][-1] for method in ("PINN", "CRVPINN")
    }
    with (output_dir / "final_metrics.json").open("w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2, ensure_ascii=False)

    # Add files for current n_train_val to overall download list
    current_run_files = (
        f"config.json",
        f"model_PINN.pt",
        f"model_CRVPINN.pt",
        f"convergence_PINN_{n_train_val}.png",
        f"convergence_CRVPINN_{n_train_val}.png",
        f"solution_slices_PINN_{n_train_val}.png",
        f"solution_slices_CRVPINN_{n_train_val}.png",
        f"centerline_comparison_{n_train_val}.png",
        f"final_metrics.json",
    )
    for filename in current_run_files:
        all_download_files.append(str(output_dir / filename))

In [ ]:
# After all experiments, save combined history and plot overall convergences
overall_output_dir = Path(initial_cfg.output_dir)
overall_output_dir.mkdir(parents=True, exist_ok=True)
save_history(all_experiment_histories, overall_output_dir)
all_download_files.append(str(overall_output_dir / "history.csv"))

In [ ]:
plot_all_convergences(all_experiment_histories, overall_output_dir)
all_download_files.append(str(overall_output_dir / "true_h1_error_comparison_all_ntrain.png"))
all_download_files.append(str(overall_output_dir / "relative_l2_error_comparison_all_ntrain.png"))

In [ ]:
plot_loss_to_true_error_ratio(all_experiment_histories, overall_output_dir)
all_download_files.append(str(overall_output_dir / "loss_to_true_error_ratio_all_ntrain.png"))

In [ ]:
print("\nAll experiments completed. Generated files:")
for f_path in all_download_files:
    print(f"  {f_path}")

In [ ]:
"""### CRVPINN Training Time Measurement

The following code runs the training of the CRVPINN model for various values of n_train and measures the time required for each iteration. The timing results will be displayed at the end.
"""

In [ ]:
n_train_values_for_timing = [12, 25, 50, 100]
crvpinn_training_times = {}

In [ ]:
initial_cfg = parse_args()

In [ ]:
print("\nRozpoczynanie pomiarów czasu treningu dla CRVPINN...")

In [ ]:
for n_train_val in n_train_values_for_timing:
    print(f"\n{'='*50}")
    print(f"Trening CRVPINN dla n_train = {n_train_val}")
    print(f"{'-'*50}")

    # Tworzenie konfiguracji dla bieżącej wartości n_train
    cfg = copy.deepcopy(initial_cfg)
    cfg.n_train = n_train_val
    # Ustaw mniejszą liczbę epok do testów czasu, aby przyspieszyć (opcjonalne)
    # cfg.epochs = 5000 # Możesz dostosować, jeśli 20000 jest zbyt długie

    set_seed(cfg.seed)
    device = select_device(cfg.device)
    if device.type == "cpu":
        torch.set_num_threads(max(1, cfg.cpu_threads))
    dtype = torch.float32 if cfg.dtype == "float32" else torch.float64
    torch.set_default_dtype(dtype)

    print(f"Urządzenie: {device}; typ danych: {dtype}")
    x, y, t, h = interior_grid(cfg.n_train, device, dtype, requires_grad=True)
    gram_inverse = SinePoissonInverse3D(cfg.n_train, h, device, dtype).to(device)

    model_crvpinn = MLP(cfg.hidden_layers, cfg.width).to(device=device, dtype=dtype)

    start_time = time.perf_counter()
    history_crvpinn = train(model_crvpinn, "CRVPINN", x, y, t, gram_inverse, cfg, device, dtype, n_train_val)
    end_time = time.perf_counter()
    training_duration = end_time - start_time
    crvpinn_training_times[n_train_val] = training_duration

    print(f"\nCRVPINN dla n_train = {n_train_val}: Czas treningu = {training_duration:.2f} sekund.")

In [ ]:
print("\n{'='*50}")
print("Podsumowanie Czasów Treningu CRVPINN:")
print(f"{'='*50}")
for n_train_val, duration in crvpinn_training_times.items():
    print(f"n_train = {n_train_val}: {duration:.2f} sekund")

In [ ]:
"""### PINN Training Time Measurement

The following code runs the training of the PINN model for various values of n_train and measures the time required for each iteration. The timing results will be displayed at the end.
"""

In [ ]:
n_train_values_for_timing = [12, 25, 50, 100]
pinn_training_times = {}

In [ ]:
initial_cfg = parse_args()

In [ ]:
print("\nRozpoczynanie pomiarów czasu treningu dla PINN...")

In [ ]:
for n_train_val in n_train_values_for_timing:
    print(f"\n{'='*50}")
    print(f"Trening PINN dla n_train = {n_train_val}")
    print(f"{'-'*50}")

    # Tworzenie konfiguracji dla bieżącej wartości n_train
    cfg = copy.deepcopy(initial_cfg)
    cfg.n_train = n_train_val
    # Ustaw mniejszą liczbę epok do testów czasu, aby przyspieszyć (opcjonalne)
    # cfg.epochs = 5000 # Możesz dostosować, jeśli 20000 jest zbyt długie

    set_seed(cfg.seed)
    device = select_device(cfg.device)
    if device.type == "cpu":
        torch.set_num_threads(max(1, cfg.cpu_threads))
    dtype = torch.float32 if cfg.dtype == "float32" else torch.float64
    torch.set_default_dtype(dtype)

    print(f"Urządzenie: {device}; typ danych: {dtype}")
    x, y, t, h = interior_grid(cfg.n_train, device, dtype, requires_grad=True)
    gram_inverse = SinePoissonInverse3D(cfg.n_train, h, device, dtype).to(device)

    model_pinn = MLP(cfg.hidden_layers, cfg.width).to(device=device, dtype=dtype)

    start_time = time.perf_counter()
    history_pinn = train(model_pinn, "PINN", x, y, t, gram_inverse, cfg, device, dtype, n_train_val)
    end_time = time.perf_counter()
    training_duration = end_time - start_time
    pinn_training_times[n_train_val] = training_duration

    print(f"\nPINN dla n_train = {n_train_val}: Czas treningu = {training_duration:.2f} sekund.")

In [ ]:
print("\n{'='*50}")
print("Podsumowanie Czasów Treningu PINN:")
print(f"{'='*50}")
for n_train_val, duration in pinn_training_times.items():
    print(f"n_train = {n_train_val}: {duration:.2f} sekund")